### Jupyter Notebook to test code in cells before adding to Python script in .py file

### Data sources:
- populations https://www.michigan-demographics.com/counties_by_population
- geopoints https://public.opendatasoft.com/explore/dataset/us-county-boundaries/export/?flg=en-us&disjunctive.statefp&disjunctive.countyfp&disjunctive.name&disjunctive.namelsad&disjunctive.stusab&disjunctive.state_name&sort=stusab
- shapefile https://public.opendatasoft.com/explore/dataset/us-county-boundaries/export/?flg=en-us&disjunctive.statefp&disjunctive.countyfp&disjunctive.name&disjunctive.namelsad&disjunctive.stusab&disjunctive.state_name&sort=stusab&refine.statefp=26

In [3]:
# Import necessary libraries
import numpy as np
import geopandas as gpd   
import pandas as pd 
from math import pi, pow, sin, cos, asin, sqrt, floor
from pulp import *

#### Precursor 1: Read in the Excel file and GeoJSON file


In [10]:
# Read in the Excel file of Michigan counties
xlsx_file_path = '/Users/stefanjenss/Desktop/DataScience/Decision_analytics/Module6/michigan_counties.xlsx'
michigan_counties = pd.read_excel(xlsx_file_path)

# Read in the GeoJSON file of Michigan county boundaries
geojson_file_path = '/Users/stefanjenss/Desktop/DataScience/Decision_analytics/Module6/michigan_counties.geojson'
michigan_counties_geojson = gpd.read_file(geojson_file_path)

# michigan_counties.head()
# michigan_counties_geojson.head()

#### Precursor 2: Merge the two data sources on country names to have a comprehensive dataset.
- In the 'michigan_counties' dataframe, the county name column is 'county_names', and in the 'michigan_counties_geojson' datafram, the county name column is 'name'.

In [18]:
# Merge the two dataframes on the county name
michigan_counties_merged = michigan_counties.merge(michigan_counties_geojson, left_on='county_names', right_on='name', how='inner')

# michigan_counties_merged.head()

#### Precursor 2: Obtain adjacency information for the countries:
- This will be done by checking which counties share a coundary using the GeoJSON file.

In [17]:
# Determine adjacency between counties based on their geometries
adjacency_matrix = []

# Loop through each county in the GeoJSON file and determine its adjacency to all other counties
for index_1, county_1 in michigan_counties_merged.iterrows(): # Loop through each county in the GeoJSON file
    neighbors = [] # Create an empty list to store the names of neighboring counties
    for index_2, county_2 in michigan_counties_merged.iterrows(): # Loop through each county in the GeoJSON file
        # Check if the two counties are the same to avoid self-comparison
        if county_1['name'] != county_2['name']:
            if county_1['geometry'].touches(county_2['geometry']): # Check if the two counties touch
                neighbors.append(county_2['name']) # If they do, add the name of the second county to the list of neighbors
    adjacency_matrix.append(neighbors) # Add the list of neighbors to the adjacency matrix

# Add adjacency information to the dataframe
michigan_counties_merged['adjacent_counties'] = adjacency_matrix

# Check the adjacency information for the first few counties
michigan_counties_merged[['name', 'adjacent_counties']].head()

,name,adjacent_counties
0,Leelanau,"[Schoolcraft, Antrim, Benzie, Grand Traverse, ..."
1,Clinton,"[Ionia, Shiawassee, Ingham, Montcalm, Eaton, G..."
2,Wexford,"[Missaukee, Osceola, Manistee, Benzie, Lake, K..."
3,Branch,"[Calhoun, St. Joseph, Kalamazoo, Hillsdale]"
4,Ionia,"[Clinton, Barry, Kent, Montcalm, Eaton, Gratiot]"


### Section 1: Calculation Distance Between County Pairs